In [ ]:
from pathlib import Path
import kagglehub


DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "meteo.csv"


if not DATA_PATH.exists():
    print("Скачиваем датасет с Kaggle...")

    kagglehub.dataset_download(
        "kapitanssb/russia-data-meteo-1966-2025",
        path="meteo.csv",
        output_dir=str(DATA_DIR)
    )

## 1. Importing Libraries

We begin by importing necessary libraries for data processing, machine learning, and automated machine learning.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# LightAutoML presets, task, and report generation
from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.dataset.roles import DatetimeRole
from lightautoml.tasks import Task
import os
from math import radians, cos, sin, asin, sqrt

## 2. Loading Data

In [2]:
df = pd.read_csv('data/meteo.csv', delimiter=';',
                 encoding='utf-8', low_memory=False, na_values=["", " "])

df = df[df['syn_stn_idx'] == 27612]

## 3. Data Augmentation and Preprocessing

In [3]:
df['year'] = df['loc_year']
df['month'] = df['loc_month']
df['day'] = df['loc_day']


needed_features = ['year', 'month', 'day', 'loc_time', 'precip_amt', 'rel_humidity', 'avg_wind_spd',
                   'max_air_temp', 'min_air_temp', 'soil_temp', 'dew_point_temp', 'total_cloud',
                   'stn_press', 'sea_level', 'part_vapor_press', 'vapor_deficit']

for feature in needed_features[4:]:
    df[feature] = df[feature].astype('str').str.strip().replace("", np.nan).astype(float)

df

,syn_stn_idx,gmt_year,gmt_month,gmt_day,gmt_time,loc_year,loc_month,loc_day,loc_time,time_period_num,...,vapor_deficit,dew_point_temp,stn_press,sea_level,bar_tend_char,bar_tend_value,bar_tend_magnitude',year,month,day
0,27612,1965,12,31,21,1966,1,1,0,1,...,86.0,0.7,-4.0,985.9,1005.6,4,0.0,1966,1,1
1,27612,1966,1,1,0,1966,1,1,3,2,...,87.0,0.7,-4.0,985.7,1005.4,7,0.2,1966,1,1
2,27612,1966,1,1,3,1966,1,1,6,3,...,90.0,0.5,-3.0,985.1,1004.7,7,0.6,1966,1,1
3,27612,1966,1,1,6,1966,1,1,9,4,...,90.0,0.5,-2.0,984.6,1004.2,7,0.5,1966,1,1
4,27612,1966,1,1,9,1966,1,1,12,5,...,88.0,0.7,-2.0,984.2,1003.8,7,0.4,1966,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175315,27612,2025,12,31,6,2025,12,31,6,4,...,NaN,NaN,NaN,NaN,,,,2025,12,31
175316,27612,2025,12,31,9,2025,12,31,9,5,...,NaN,NaN,NaN,NaN,,,,2025,12,31
175317,27612,2025,12,31,12,2025,12,31,12,6,...,NaN,NaN,NaN,NaN,,,,2025,12,31
175318,27612,2025,12,31,15,2025,12,31,15,7,...,NaN,NaN,NaN,NaN,,,,2025,12,31


In [4]:
df_train = df[df['year'] <= 2010][needed_features]
df_val = df[(df['year'] > 2010) & (df['year'] <= 2024)][needed_features]

In [5]:
df_train = df_train.groupby(['year', 'month', 'day']).agg({
    'precip_amt': 'sum',
    'rel_humidity': 'mean',
    'avg_wind_spd': 'mean',
    'max_air_temp' : 'max',
    'min_air_temp' : 'min',
    'soil_temp': 'mean',
    'dew_point_temp': 'mean',
    'total_cloud': 'mean',
    'stn_press': 'mean',
    'part_vapor_press': 'mean',
    'vapor_deficit': 'mean'
})

In [6]:
df_val = df_val.groupby(['year', 'month', 'day']).agg({
    'precip_amt': 'sum',
    'rel_humidity': 'mean',
    'avg_wind_spd': 'mean',
    'max_air_temp' : 'max',
    'min_air_temp' : 'min',
    'soil_temp': 'mean',
    'dew_point_temp': 'mean',
    'total_cloud': 'mean',
    'stn_press': 'mean',
    'part_vapor_press': 'mean',
    'vapor_deficit': 'mean'
})

In [7]:
features_to_fill = ['rel_humidity', 'avg_wind_spd', 'max_air_temp',  'min_air_temp', 'soil_temp', 'dew_point_temp',
                    'total_cloud', 'stn_press', 'part_vapor_press', 'vapor_deficit']

df_train[features_to_fill] = df_train[features_to_fill].interpolate(method='linear', limit=3, limit_direction="forward")

In [8]:
seasonal_means = df_train.groupby(["month", "day"])[features_to_fill].mean()
seasonal_means

rel_humidity  avg_wind_spd  max_air_temp  min_air_temp  soil_temp  \
month day                                                                      
1     1        3.339559      1.963889     -6.629412    -10.194118  -8.709722   
      2        3.192169      1.933333     -6.850000    -10.038235  -8.938889   
      3        3.256213      1.741667     -6.502941    -10.641176  -9.493889   
      4        3.362206      1.783333     -6.323529     -9.973529  -8.716389   
      5        3.374485      1.841667     -6.052941     -9.279412  -8.455833   
...                 ...           ...           ...           ...        ...   
12    27       3.658607      1.711111     -4.968571     -8.228571  -7.600397   
      28       3.732786      1.800000     -4.925714     -8.308571  -7.521944   
      29       3.549286      1.908333     -5.205714     -9.177143  -7.681667   
      30       3.859321      1.947222     -4.428571     -8.725714  -7.461944   
      31       3.654429      2.141667     -4.985714     -9.222857  -7.315000   

           dew_point_temp  total_cloud  stn_press  part_vapor_press  \
month day                                                             
1     1          0.648444     8.438889  -9.801944         -8.149755   
      2          0.642944     8.463889 -10.237778         -7.925956   
      3          0.550758     8.544444 -10.320833         -8.145588   
      4          0.557472     8.319444  -9.837500         -7.812500   
      5          0.626806     8.550000  -9.672222         -7.215441   
...                   ...          ...        ...               ...   
12    27         0.536250     8.722222  -8.299167         -6.314643   
      28         0.566389     8.600000  -8.319444         -6.283571   
      29         0.632139     8.866667  -9.127222         -6.890714   
      30         0.590750     8.983333  -8.426667         -5.971786   
      31         0.614111     9.133333  -8.412976         -6.496071   

           vapor_deficit  
month day                 
1     1        83.152778  
      2        82.419444  
      3        83.983333  
      4        83.780556  
      5        83.141667  
...                  ...  
12    27       86.102778  
      28       85.952778  
      29       83.883333  
      30       85.291667  
      31       84.844444  

[366 rows x 10 columns]

In [9]:
df_train[features_to_fill] = df_train[features_to_fill].fillna(seasonal_means)
df_val[features_to_fill] = df_val[features_to_fill].fillna(seasonal_means)

In [10]:
def classify_rain(value):
    try:
        return 1 if float(value) > 0.0 else 0
    except ValueError:
        return 0

df_train['rain'] = df_train['precip_amt'].apply(classify_rain)
df_val['rain'] = df_val['precip_amt'].apply(classify_rain)

df_train[['precip_amt', 'rain']]

precip_amt  rain
year month day                  
1966 1     1           1.8     1
           2           0.2     1
           3           0.4     1
           4           0.2     1
           5           1.6     1
...                    ...   ...
2010 12    27          0.3     1
           28          0.5     1
           29          9.4     1
           30          1.1     1
           31          2.1     1

[16436 rows x 2 columns]

In [11]:
df_train['rain_shift1'] = df_train['rain'].shift(1)
df_val['rain_shift1'] = df_val['rain'].shift(1)

In [12]:
def classify_precip(value):
    if value == 0.0:
        return 0
    elif value < 3.0:
        return 1
    elif value < 10.0:
        return 2
    return 3


df_train['precip'] = df_train['precip_amt'].apply(classify_precip)
df_val['precip'] = df_val['precip_amt'].apply(classify_precip)

In [13]:
df_train['precip']

year  month  day
1966  1      1      1
             2      1
             3      1
             4      1
             5      1
                   ..
2010  12     27     1
             28     1
             29     2
             30     1
             31     1
Name: precip, Length: 16436, dtype: int64

In [14]:
df_train = df_train[['rain', 'precip']]
df_val = df_val[['rain', 'precip']]
df_train

rain  precip
year month day              
1966 1     1       1       1
           2       1       1
           3       1       1
           4       1       1
           5       1       1
...              ...     ...
2010 12    27      1       1
           28      1       1
           29      1       2
           30      1       1
           31      1       1

[16436 rows x 2 columns]

In [15]:
for shft in range(1, 31):
    df_train['precip_shift' + str(shft)] = df_train['precip'].shift(shft)


df_train

rain  precip  precip_shift1  precip_shift2  precip_shift3  \
year month day                                                              
1966 1     1       1       1            NaN            NaN            NaN   
           2       1       1            1.0            NaN            NaN   
           3       1       1            1.0            1.0            NaN   
           4       1       1            1.0            1.0            1.0   
           5       1       1            1.0            1.0            1.0   
...              ...     ...            ...            ...            ...   
2010 12    27      1       1            2.0            2.0            2.0   
           28      1       1            1.0            2.0            2.0   
           29      1       2            1.0            1.0            2.0   
           30      1       1            2.0            1.0            1.0   
           31      1       1            1.0            2.0            1.0   

                precip_shift4  precip_shift5  precip_shift6  precip_shift7  \
year month day                                                               
1966 1     1              NaN            NaN            NaN            NaN   
           2              NaN            NaN            NaN            NaN   
           3              NaN            NaN            NaN            NaN   
           4              NaN            NaN            NaN            NaN   
           5              1.0            NaN            NaN            NaN   
...                       ...            ...            ...            ...   
2010 12    27             1.0            0.0            1.0            2.0   
           28             2.0            1.0            0.0            1.0   
           29             2.0            2.0            1.0            0.0   
           30             2.0            2.0            2.0            1.0   
           31             1.0            2.0            2.0            2.0   

                precip_shift8  ...  precip_shift21  precip_shift22  \
year month day                 ...                                   
1966 1     1              NaN  ...             NaN             NaN   
           2              NaN  ...             NaN             NaN   
           3              NaN  ...             NaN             NaN   
           4              NaN  ...             NaN             NaN   
           5              NaN  ...             NaN             NaN   
...                       ...  ...             ...             ...   
2010 12    27             1.0  ...             0.0             0.0   
           28             2.0  ...             1.0             0.0   
           29             1.0  ...             1.0             1.0   
           30             0.0  ...             2.0             1.0   
           31             1.0  ...             3.0             2.0   

                precip_shift23  precip_shift24  precip_shift25  \
year month day                                                   
1966 1     1               NaN             NaN             NaN   
           2               NaN             NaN             NaN   
           3               NaN             NaN             NaN   
           4               NaN             NaN             NaN   
           5               NaN             NaN             NaN   
...                        ...             ...             ...   
2010 12    27              2.0             0.0             0.0   
           28              0.0             2.0             0.0   
           29              0.0             0.0             2.0   
           30              1.0             0.0             0.0   
           31              1.0             1.0             0.0   

                precip_shift26  precip_shift27  precip_shift28  \
year month day                                                   
1966 1     1               NaN             NaN             NaN   
           2               NaN 

In [16]:
for shft in range(1, 31):
    df_val['precip_shift' + str(shft)] = df_val['precip'].shift(shft)


df_val

rain  precip  precip_shift1  precip_shift2  precip_shift3  \
year month day                                                              
2011 1     1       1       1            NaN            NaN            NaN   
           2       1       2            1.0            NaN            NaN   
           3       1       1            2.0            1.0            NaN   
           4       0       0            1.0            2.0            1.0   
           5       0       0            0.0            1.0            2.0   
...              ...     ...            ...            ...            ...   
2024 12    27      1       1            0.0            1.0            1.0   
           28      0       0            1.0            0.0            1.0   
           29      0       0            0.0            1.0            0.0   
           30      1       1            0.0            0.0            1.0   
           31      1       2            1.0            0.0            0.0   

                precip_shift4  precip_shift5  precip_shift6  precip_shift7  \
year month day                                                               
2011 1     1              NaN            NaN            NaN            NaN   
           2              NaN            NaN            NaN            NaN   
           3              NaN            NaN            NaN            NaN   
           4              NaN            NaN            NaN            NaN   
           5              1.0            NaN            NaN            NaN   
...                       ...            ...            ...            ...   
2024 12    27             0.0            0.0            1.0            1.0   
           28             1.0            0.0            0.0            1.0   
           29             1.0            1.0            0.0            0.0   
           30             0.0            1.0            1.0            0.0   
           31             1.0            0.0            1.0            1.0   

                precip_shift8  ...  precip_shift21  precip_shift22  \
year month day                 ...                                   
2011 1     1              NaN  ...             NaN             NaN   
           2              NaN  ...             NaN             NaN   
           3              NaN  ...             NaN             NaN   
           4              NaN  ...             NaN             NaN   
           5              NaN  ...             NaN             NaN   
...                       ...  ...             ...             ...   
2024 12    27             1.0  ...             1.0             0.0   
           28             1.0  ...             2.0             1.0   
           29             1.0  ...             3.0             2.0   
           30             0.0  ...             2.0             3.0   
           31             0.0  ...             0.0             2.0   

                precip_shift23  precip_shift24  precip_shift25  \
year month day                                                   
2011 1     1               NaN             NaN             NaN   
           2               NaN             NaN             NaN   
           3               NaN             NaN             NaN   
           4               NaN             NaN             NaN   
           5               NaN             NaN             NaN   
...                        ...             ...             ...   
2024 12    27              1.0             1.0             0.0   
           28              0.0             1.0             1.0   
           29              1.0             0.0             1.0   
           30              2.0             1.0             0.0   
           31              3.0             2.0             1.0   

                precip_shift26  precip_shift27  precip_shift28  \
year month day                                                   
2011 1     1               NaN             NaN             NaN   
           2               NaN 

In [17]:
df_train = df_train.dropna()
df_val = df_val.dropna()
df_train

rain  precip  precip_shift1  precip_shift2  precip_shift3  \
year month day                                                              
1966 1     31      1       2            2.0            0.0            0.0   
     2     1       0       0            2.0            2.0            0.0   
           2       1       1            0.0            2.0            2.0   
           3       0       0            1.0            0.0            2.0   
           4       0       0            0.0            1.0            0.0   
...              ...     ...            ...            ...            ...   
2010 12    27      1       1            2.0            2.0            2.0   
           28      1       1            1.0            2.0            2.0   
           29      1       2            1.0            1.0            2.0   
           30      1       1            2.0            1.0            1.0   
           31      1       1            1.0            2.0            1.0   

                precip_shift4  precip_shift5  precip_shift6  precip_shift7  \
year month day                                                               
1966 1     31             0.0            1.0            1.0            0.0   
     2     1              0.0            0.0            1.0            1.0   
           2              0.0            0.0            0.0            1.0   
           3              2.0            0.0            0.0            0.0   
           4              2.0            2.0            0.0            0.0   
...                       ...            ...            ...            ...   
2010 12    27             1.0            0.0            1.0            2.0   
           28             2.0            1.0            0.0            1.0   
           29             2.0            2.0            1.0            0.0   
           30             2.0            2.0            2.0            1.0   
           31             1.0            2.0            2.0            2.0   

                precip_shift8  ...  precip_shift21  precip_shift22  \
year month day                 ...                                   
1966 1     31             2.0  ...             0.0             0.0   
     2     1              0.0  ...             1.0             0.0   
           2              1.0  ...             1.0             1.0   
           3              1.0  ...             2.0             1.0   
           4              0.0  ...             1.0             2.0   
...                       ...  ...             ...             ...   
2010 12    27             1.0  ...             0.0             0.0   
           28             2.0  ...             1.0             0.0   
           29             1.0  ...             1.0             1.0   
           30             0.0  ...             2.0             1.0   
           31             1.0  ...             3.0             2.0   

                precip_shift23  precip_shift24  precip_shift25  \
year month day                                                   
1966 1     31              2.0             2.0             2.0   
     2     1               0.0             2.0             2.0   
           2               0.0             0.0             2.0   
           3               1.0             0.0             0.0   
           4               1.0             1.0             0.0   
...                        ...             ...             ...   
2010 12    27              2.0             0.0             0.0   
           28              0.0             2.0             0.0   
           29              0.0             0.0             2.0   
           30              1.0             0.0             0.0   
           31              1.0             1.0             0.0   

                precip_shift26  precip_shift27  precip_shift28  \
year month day                                                   
1966 1     31              1.0             1.0             1.0   
     2     1               2.0 

In [18]:
df_train = df_train.reset_index()
df_val = df_val.reset_index()
df_train['date'] = pd.to_datetime(df_train[['year', 'month', 'day']])
df_val['date'] = pd.to_datetime(df_val[['year', 'month', 'day']])

In [19]:
df_train

,year,month,day,rain,precip,precip_shift1,precip_shift2,precip_shift3,precip_shift4,precip_shift5,...,precip_shift22,precip_shift23,precip_shift24,precip_shift25,precip_shift26,precip_shift27,precip_shift28,precip_shift29,precip_shift30,date
0,1966,1,31,1,2,2.0,0.0,0.0,0.0,1.0,...,0.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1966-01-31
1,1966,2,1,0,0,2.0,2.0,0.0,0.0,0.0,...,0.0,0.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1966-02-01
2,1966,2,2,1,1,0.0,2.0,2.0,0.0,0.0,...,1.0,0.0,0.0,2.0,2.0,2.0,1.0,1.0,1.0,1966-02-02
3,1966,2,3,0,0,1.0,0.0,2.0,2.0,0.0,...,1.0,1.0,0.0,0.0,2.0,2.0,2.0,1.0,1.0,1966-02-03
4,1966,2,4,0,0,0.0,1.0,0.0,2.0,2.0,...,2.0,1.0,1.0,0.0,0.0,2.0,2.0,2.0,1.0,1966-02-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16401,2010,12,27,1,1,2.0,2.0,2.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2010-12-27
16402,2010,12,28,1,1,1.0,2.0,2.0,2.0,1.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,2010-12-28
16403,2010,12,29,1,2,1.0,1.0,2.0,2.0,2.0,...,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2010-12-29
16404,2010,12,30,1,1,2.0,1.0,1.0,2.0,2.0,...,1.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,2010-12-30


In [20]:
df_train = df_train.drop(columns=['year', 'month', 'day'])
df_val = df_val.drop(columns=['year', 'month', 'day'])

In [21]:
df_train

,rain,precip,precip_shift1,precip_shift2,precip_shift3,precip_shift4,precip_shift5,precip_shift6,precip_shift7,precip_shift8,...,precip_shift22,precip_shift23,precip_shift24,precip_shift25,precip_shift26,precip_shift27,precip_shift28,precip_shift29,precip_shift30,date
0,1,2,2.0,0.0,0.0,0.0,1.0,1.0,0.0,2.0,...,0.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1966-01-31
1,0,0,2.0,2.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1966-02-01
2,1,1,0.0,2.0,2.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,2.0,2.0,2.0,1.0,1.0,1.0,1966-02-02
3,0,0,1.0,0.0,2.0,2.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,0.0,2.0,2.0,2.0,1.0,1.0,1966-02-03
4,0,0,0.0,1.0,0.0,2.0,2.0,0.0,0.0,0.0,...,2.0,1.0,1.0,0.0,0.0,2.0,2.0,2.0,1.0,1966-02-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16401,1,1,2.0,2.0,2.0,1.0,0.0,1.0,2.0,1.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2010-12-27
16402,1,1,1.0,2.0,2.0,2.0,1.0,0.0,1.0,2.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,2010-12-28
16403,1,2,1.0,1.0,2.0,2.0,2.0,1.0,0.0,1.0,...,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2010-12-29
16404,1,1,2.0,1.0,1.0,2.0,2.0,2.0,1.0,0.0,...,1.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,2010-12-30


In [22]:
df_val

,rain,precip,precip_shift1,precip_shift2,precip_shift3,precip_shift4,precip_shift5,precip_shift6,precip_shift7,precip_shift8,...,precip_shift22,precip_shift23,precip_shift24,precip_shift25,precip_shift26,precip_shift27,precip_shift28,precip_shift29,precip_shift30,date
0,1,1,2.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,2.0,1.0,2011-01-31
1,1,1,1.0,2.0,1.0,1.0,1.0,0.0,1.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,2.0,2011-02-01
2,1,1,1.0,1.0,2.0,1.0,1.0,1.0,0.0,1.0,...,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,2011-02-02
3,1,2,1.0,1.0,1.0,2.0,1.0,1.0,1.0,0.0,...,0.0,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2011-02-03
4,1,1,2.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,1.0,0.0,1.0,0.0,2011-02-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5079,1,1,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,2024-12-27
5080,0,0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,2024-12-28
5081,0,0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,2.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,2024-12-29
5082,1,1,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,3.0,2.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,2024-12-30


## 4. Modeling with LightAutoML

### Model Setup

Define key parameters for our model.

In [23]:
N_THREADS = 4
N_FOLDS = 10
TIMEOUT = 60 * 120  # 2 hours
TARGET_NAME = 'rain'
DATE_COLUMN = 'date'
task = Task('binary', metric='auc')
roles = {
         'target': TARGET_NAME, 
         DatetimeRole(
         seasonality=('d', 'm'),
         base_date=True
        ): DATE_COLUMN
        }

### Model Initialization and Training

Use `LightAutoML` to initialize and train a regression model with specified settings.

In [24]:
for shft in range(1, 31):
    df_train['precip_shift' + str(shft)] = df_train['precip_shift' + str(shft)].astype(int)

for shft in range(1, 31):
    df_val['precip_shift' + str(shft)] = df_val['precip_shift' + str(shft)].astype(int)


df_train

,rain,precip,precip_shift1,precip_shift2,precip_shift3,precip_shift4,precip_shift5,precip_shift6,precip_shift7,precip_shift8,...,precip_shift22,precip_shift23,precip_shift24,precip_shift25,precip_shift26,precip_shift27,precip_shift28,precip_shift29,precip_shift30,date
0,1,2,2,0,0,0,1,1,0,2,...,0,2,2,2,1,1,1,1,1,1966-01-31
1,0,0,2,2,0,0,0,1,1,0,...,0,0,2,2,2,1,1,1,1,1966-02-01
2,1,1,0,2,2,0,0,0,1,1,...,1,0,0,2,2,2,1,1,1,1966-02-02
3,0,0,1,0,2,2,0,0,0,1,...,1,1,0,0,2,2,2,1,1,1966-02-03
4,0,0,0,1,0,2,2,0,0,0,...,2,1,1,0,0,2,2,2,1,1966-02-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16401,1,1,2,2,2,1,0,1,2,1,...,0,2,0,0,0,0,0,1,0,2010-12-27
16402,1,1,1,2,2,2,1,0,1,2,...,0,0,2,0,0,0,0,0,1,2010-12-28
16403,1,2,1,1,2,2,2,1,0,1,...,1,0,0,2,0,0,0,0,0,2010-12-29
16404,1,1,2,1,1,2,2,2,1,0,...,1,1,0,0,2,0,0,0,0,2010-12-30


In [ ]:
df_val.to_csv("data/topredict_precip.csv")

In [25]:
df_train = df_train.drop(columns='precip')
df_val = df_val.drop(columns='precip')
df_train

,rain,precip_shift1,precip_shift2,precip_shift3,precip_shift4,precip_shift5,precip_shift6,precip_shift7,precip_shift8,precip_shift9,...,precip_shift22,precip_shift23,precip_shift24,precip_shift25,precip_shift26,precip_shift27,precip_shift28,precip_shift29,precip_shift30,date
0,1,2,0,0,0,1,1,0,2,2,...,0,2,2,2,1,1,1,1,1,1966-01-31
1,0,2,2,0,0,0,1,1,0,2,...,0,0,2,2,2,1,1,1,1,1966-02-01
2,1,0,2,2,0,0,0,1,1,0,...,1,0,0,2,2,2,1,1,1,1966-02-02
3,0,1,0,2,2,0,0,0,1,1,...,1,1,0,0,2,2,2,1,1,1966-02-03
4,0,0,1,0,2,2,0,0,0,1,...,2,1,1,0,0,2,2,2,1,1966-02-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16401,1,2,2,2,1,0,1,2,1,1,...,0,2,0,0,0,0,0,1,0,2010-12-27
16402,1,1,2,2,2,1,0,1,2,1,...,0,0,2,0,0,0,0,0,1,2010-12-28
16403,1,1,1,2,2,2,1,0,1,2,...,1,0,0,2,0,0,0,0,0,2010-12-29
16404,1,2,1,1,2,2,2,1,0,1,...,1,1,0,0,2,0,0,0,0,2010-12-30


In [26]:
automl = TabularAutoML(
    task=task,
    timeout=TIMEOUT,
    cpu_limit=N_THREADS,
    general_params={'use_algos': [['linear_l2', 'lgb', 'lgb_tuned', 'cb', 'cb_tuned']]},
    reader_params={'n_jobs': N_THREADS, 'cv': N_FOLDS, 'random_state': 42}
)

pred_tr = automl.fit_predict(df_train, roles=roles, verbose=1, path_to_save="model/laml_model.joblib")

[15:18:06] Stdout logging level is INFO.
[15:18:06] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer
[15:18:06] Task: binary

[15:18:06] Start automl preset with listed constraints:
[15:18:06] - time: 7200.00 seconds
[15:18:06] - CPU: 4 cores
[15:18:06] - memory: 16 GB

[15:18:06] Train data shape: (16406, 32)

[15:18:09] Layer 1 train process start. Time left 7196.21 secs
[15:18:09] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[15:18:13] Fitting Lvl_0_Pipe_0_Mod_0_LinearL2 finished. score = 0.6912713905881418
[15:18:13] Lvl_0_Pipe_0_Mod_0_LinearL2 fitting and predicting completed
[15:18:13] Time left 7192.78 secs

[15:18:13] Selector_LightGBM fitting and predicting completed
[15:18:13] Start fitting Lvl_0_Pipe_1_Mod_0_LightGBM ...
[15:18:17] Fitting Lvl_0_Pipe_1_Mod_0_LightGBM finished. score = 0.6876434379898382
[15:18:17] Lvl_0_Pipe_1_Mod_0_LightGBM fitting and predicting completed
[15:18:17] Start hyperparameters optimization for L

Optimization Progress: 100%|██████████| 101/101 [00:36<00:00,  2.77it/s, best_trial=71, best_value=0.695]

[15:18:54] Hyperparameters optimization for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM completed
[15:18:54] Start fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM ...


[15:18:55] Fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM finished. score = 0.6872741762655369
[15:18:55] Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM fitting and predicting completed
[15:18:55] Start fitting Lvl_0_Pipe_1_Mod_2_CatBoost ...
[15:18:59] Fitting Lvl_0_Pipe_1_Mod_2_CatBoost finished. score = 0.695398721887172
[15:18:59] Lvl_0_Pipe_1_Mod_2_CatBoost fitting and predicting completed
[15:18:59] Start hyperparameters optimization for Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost ... Time budget is 300.00 secs


Optimization Progress: 100%|██████████| 101/101 [00:45<00:00,  2.23it/s, best_trial=12, best_value=0.691]

[15:19:45] Hyperparameters optimization for Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost completed
[15:19:45] Start fitting Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost ...


[15:19:50] Fitting Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost finished. score = 0.6904466971533244
[15:19:50] Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost fitting and predicting completed
[15:19:50] Time left 7095.65 secs

[15:19:50] Layer 1 training completed.

[15:19:50] Blending: optimization starts with equal weights. Score = 0.6960542
[15:19:50] Blending: iteration 0: score = 0.6967686, weights = [0.15920472 0.22686681 0.         0.5269807  0.0869478 ]
[15:19:50] Blending: iteration 1: score = 0.6967695, weights = [0.16227636 0.235615   0.         0.51222825 0.08988038]
[15:19:50] Blending: iteration 2: score = 0.6967710, weights = [0.15177599 0.22329439 0.         0.5610636  0.06386603]
[15:19:50] Blending: no improvements for score. Terminated.

[15:19:50] Blending: best score = 0.6967710, best weights = [0.15177599 0.22329439 0.         0.5610636  0.06386603]
[15:19:51] Automl preset training completed in 104.92 seconds

[15:19:51] Model description:
Final prediction for new objects (level 0) = 
	

## 5. Making Predictions

Generate predictions for the test data

In [27]:
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

In [28]:
pred_proba = automl.predict(df_val).data[:, 0]
true_rain = df_val['rain'].values

In [29]:
val_auc = roc_auc_score(true_rain, pred_proba)
val_auc

0.7129151660276766

In [30]:
pred_t = (pred_proba > 0.5).astype(int)
accuracy_score(true_rain, pred_t)

0.6675845790715972

In [31]:
best_acc_threshold, best_acc = 0.5, 0.0

for threshold in np.arange(0.1, 0.9, 0.05):
    pred = (pred_proba > threshold).astype(int)
    acc = accuracy_score(true_rain, pred)
    if acc > best_acc:
        best_acc_threshold = threshold
        best_acc = acc

print(f"Accuracy: {best_acc} using threshold of {best_acc_threshold}")

Accuracy: 0.6675845790715972 using threshold of 0.5000000000000001


In [32]:
print(classification_report(true_rain, pred_t, target_names=['No Rain', 'Rain']))

              precision    recall  f1-score   support

     No Rain       0.66      0.64      0.65      2455
        Rain       0.67      0.70      0.68      2629

    accuracy                           0.67      5084
   macro avg       0.67      0.67      0.67      5084
weighted avg       0.67      0.67      0.67      5084



In [33]:
df_val.to_csv("data/topredict.csv", index=False)